In [1]:
import numpy as np
import pandas as pd

from scipy import stats
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

import format_chronic_stim
import stim_plots
import waveform_analysis

import sys
sys.path.append("../utils/")
import load_matlab_data

import scipy.io
import os

In [2]:
''' set paths '''
# set parent directory
root_dir = "Z:/Isabel/data/hpc_implants/"

# session params
bird_id = "LIM63"
session_id = "240610"
ephys_id = "LIM63_240610_131820"
stim_pol = "neg_150"
only_good = True

# data dir
session_dir = f"{root_dir}{bird_id}/{bird_id}_{session_id}/{ephys_id}/"
stim_dir = f"{session_dir}raw_ephys_output/"
ks_dir = f"{session_dir}kilosort4/"

# set save folder
save_folder = f"../figures/antidromic_hpc_to_lhy/{bird_id}/{bird_id}_{session_id}/waveform_matching/"
if os.path.isdir(save_folder):
    print('save folder exists')
else:
    os.mkdir(save_folder)

In [37]:
# waveform params
n_wf = 1e3 # number of waveforms per unit
spk_pre = 1.5e-3 # seconds before spike time to save
spk_dur = 4e-3 # seconds; duration of waveform

In [42]:
# intan header
intan_info = load_matlab_data.loadmat_sbx(f"{stim_dir}intan_info.mat")
intan_info = intan_info['header']
sampling_rate = intan_info['sample_rate']

# get the channel IDs
amp_ch_info_mat = intan_info['amplifier_channels']
ch_sort_idx = np.asarray([])
ch_names_unsorted = []
for amp_ch in amp_ch_info_mat:
    for strg in amp_ch._fieldnames:
        if strg == 'custom_channel_name':
            name = amp_ch.__dict__[strg]
            ch_names_unsorted.append(name)
        elif strg == 'custom_order':
            idx = amp_ch.__dict__[strg]
            ch_sort_idx = np.append(ch_sort_idx, idx)
ch_sort_idx = ch_sort_idx.astype(int)

Z:/Isabel/data/hpc_implants/SPP134/SPP134_240702/SPP134_240702_143210/raw_ephys_output/intan_info.mat


In [27]:
''' get info from kilosort outputs '''
# not sure whether this or the intan header (or both?) is correct
ch_map = np.load(f"{ks_dir}channel_map.npy") 

# phy info
templates = np.load(f"{ks_dir}templates.npy")
ch_best = (templates**2).sum(axis=1).argmax(axis=-1)
phy_info = pd.read_csv(f"{ks_dir}cluster_KSLabel.tsv", sep='\t')
cluster_id = phy_info['cluster_id'].values
ks_label = phy_info['KSLabel'].values

# spike info
amplitudes = np.load(f"{ks_dir}amplitudes.npy")
st = np.load(f"{ks_dir}spike_times.npy")
clu = np.load(f"{ks_dir}spike_clusters.npy")

In [43]:
templates.shape

(192, 61, 64)

In [38]:
# only keep good units or include mua
if only_good:
    good_idx = (ks_label == 'good').astype(bool)
    good_clusters = cluster_id[good_idx]
else:
    good_clusters = cluster_id

In [47]:
# preallocate variables
n_units = good_clusters.shape[0]
n_ch = templates.shape[-1]
spk_dur_samples = int(spk_dur * sampling_rate)

avg_wf = np.zeros((n_units, spk_dur_samples, n_ch))

In [ ]:
# get the spikes associated with each unit
for c in good_clusters: